In [ ]:
print("=" * 80)
print("TRAINING THREE MODELS")
print("=" * 80)

# ============================================================================
# Model 1: Traditional ML - Random Forest
# ============================================================================
print("\n[1/3] Training Random Forest (Traditional ML)...")
try:
    rf_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        random_state=42,
        n_jobs=-1,
        verbose=0
    )
    rf_model.fit(X_train_scaled, y_train)
    rf_pred = rf_model.predict(X_test_scaled)
    print("  ✓ Random Forest trained successfully")
except Exception as e:
    print(f"  ✗ Error: {e}")
    rf_model = None
    rf_pred = None

# ============================================================================
# Model 2: Advanced ML - XGBoost
# ============================================================================
print("\n[2/3] Training XGBoost (Advanced ML / Gradient Boosting)...")
try:
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        use_label_encoder=False,
        eval_metric='mlogloss',
        verbosity=0
    )
    xgb_model.fit(X_train_scaled, y_train, eval_set=[(X_val_scaled, y_val)], verbose=0)
    xgb_pred = xgb_model.predict(X_test_scaled)
    print("  ✓ XGBoost trained successfully")
except Exception as e:
    print(f"  ✗ Error: {e}")
    xgb_model = None
    xgb_pred = None

# ============================================================================
# Model 3: Deep Learning - Neural Network
# ============================================================================
print("\n[3/3] Training Neural Network (Deep Learning / Keras)...")
try:
    # Build neural network
    dl_model = keras.Sequential([
        layers.Dense(128, activation='relu', input_shape=(n_features,), name='input_layer'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        layers.Dense(64, activation='relu', name='hidden_1'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        layers.Dense(32, activation='relu', name='hidden_2'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        layers.Dense(16, activation='relu', name='hidden_3'),
        
        layers.Dense(len(np.unique(y_train)), activation='softmax', name='output_layer')
    ])
    
    dl_model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Train
    history = dl_model.fit(
        X_train_scaled, y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=30,
        batch_size=32,
        verbose=0
    )
    
    dl_pred = np.argmax(dl_model.predict(X_test_scaled, verbose=0), axis=1)
    print("  ✓ Deep Learning model trained successfully")
    
except Exception as e:
    print(f"  ✗ Error: {e}")
    dl_model = None
    dl_pred = None
    history = None

print("\n" + "=" * 80)
print("✓ All models trained!")
print("=" * 80)

In [ ]:
# Compute metrics for all models
def compute_metrics(y_true, y_pred, model_name):
    """Compute classification metrics."""
    return {
        'model': model_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall': recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'confusion_matrix': confusion_matrix(y_true, y_pred)
    }

metrics_dict = {}

if rf_pred is not None:
    metrics_dict['Random Forest'] = compute_metrics(y_test, rf_pred, 'Random Forest')

if xgb_pred is not None:
    metrics_dict['XGBoost'] = compute_metrics(y_test, xgb_pred, 'XGBoost')

if dl_pred is not None:
    metrics_dict['Deep Learning'] = compute_metrics(y_test, dl_pred, 'Deep Learning')

# ============================================================================
# Visualization 1: Metrics Comparison
# ============================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Model Performance Comparison - Classification Metrics', fontsize=16, fontweight='bold', y=1.00)

models = list(metrics_dict.keys())
colors = ['#FF6B6B', '#4ECDC4', '#95E1D3']

# 1.1 Accuracy
ax = axes[0, 0]
accuracies = [metrics_dict[m]['accuracy'] for m in models]
bars = ax.bar(models, accuracies, color=colors[:len(models)], alpha=0.8, edgecolor='black', linewidth=2)
ax.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
ax.set_title('Accuracy Comparison', fontsize=12, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, accuracies):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

# 1.2 Precision
ax = axes[0, 1]
precisions = [metrics_dict[m]['precision'] for m in models]
bars = ax.bar(models, precisions, color=colors[:len(models)], alpha=0.8, edgecolor='black', linewidth=2)
ax.set_ylabel('Precision (Macro)', fontsize=11, fontweight='bold')
ax.set_title('Precision Comparison', fontsize=12, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, precisions):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

# 1.3 Recall
ax = axes[1, 0]
recalls = [metrics_dict[m]['recall'] for m in models]
bars = ax.bar(models, recalls, color=colors[:len(models)], alpha=0.8, edgecolor='black', linewidth=2)
ax.set_ylabel('Recall (Macro)', fontsize=11, fontweight='bold')
ax.set_title('Recall Comparison', fontsize=12, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, recalls):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

# 1.4 F1 Score
ax = axes[1, 1]
f1_scores = [metrics_dict[m]['f1'] for m in models]
bars = ax.bar(models, f1_scores, color=colors[:len(models)], alpha=0.8, edgecolor='black', linewidth=2)
ax.set_ylabel('F1 Score (Macro)', fontsize=11, fontweight='bold')
ax.set_title('F1 Score Comparison', fontsize=12, fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, f1_scores):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('reports/figures/01_metrics_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Metrics comparison visualization saved!")

# ============================================================================
# Visualization 2: Confusion Matrices
# ============================================================================
fig, axes = plt.subplots(1, len(models), figsize=(5*len(models), 4))
fig.suptitle('Confusion Matrices - All Models', fontsize=15, fontweight='bold')

if len(models) == 1:
    axes = [axes]

for idx, (ax, model_name) in enumerate(zip(axes, models)):
    cm = metrics_dict[model_name]['confusion_matrix']
    
    # Normalize for visualization
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Plot heatmap
    im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
    
    # Add text
    tick_marks = np.arange(cm.shape[0])
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    ax.set_xlabel('Predicted Label', fontsize=10, fontweight='bold')
    ax.set_ylabel('True Label', fontsize=10, fontweight='bold')
    ax.set_title(model_name, fontsize=12, fontweight='bold')
    
    # Add values in cells
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            text_color = 'white' if cm_norm[i, j] > 0.5 else 'black'
            ax.text(j, i, f'{cm[i, j]}\n({cm_norm[i, j]:.2f})',
                   ha='center', va='center', color=text_color, fontsize=9, fontweight='bold')
    
    plt.colorbar(im, ax=ax, label='Normalized Count')

plt.tight_layout()
plt.savefig('reports/figures/02_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Confusion matrices visualization saved!")

In [ ]:
# ============================================================================
# Visualization 3: Neural Network Architecture Diagram
# ============================================================================

if dl_model is not None:
    fig, ax = plt.subplots(figsize=(14, 11))
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 13)
    ax.axis('off')
    
    # Title
    ax.text(5, 12.5, 'Deep Learning Model Architecture',
           ha='center', fontsize=18, fontweight='bold')
    ax.text(5, 12.0, 'Keras Sequential Neural Network with Batch Normalization & Dropout',
           ha='center', fontsize=11, style='italic', color='gray')
    
    # ===== INPUT LAYER =====
    layer_y = 11.0
    layer_spacing = 2.0
    
    # Input layer
    ax.text(0.3, layer_y + 0.5, 'INPUT', fontsize=10, fontweight='bold')
    draw_layer = lambda x, y, nodes, color, label: [
        [plt.Circle((x, y - i*0.3), 0.2, color=color, ec='black', linewidth=1.5) 
         for ax.add_patch(plt.Circle((x, y - i*0.3), 0.2, color=color, ec='black', linewidth=1.5))
         for i in range(nodes)],
        ax.text(x + 0.8, y, label, fontsize=9, fontweight='bold', va='center')
    ]
    
    for i in range(min(5, n_features)):
        circle = plt.Circle((1.5, layer_y - i*0.25), 0.18, color='#FF6B6B', ec='black', linewidth=1.5)
        ax.add_patch(circle)
    
    ax.text(1.5, layer_y - 1.3, f'Input: {n_features} features', 
           ha='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='#FFE0E0', alpha=0.7))
    
    # ===== HIDDEN LAYER 1 =====
    layer_y -= layer_spacing
    ax.text(0.3, layer_y + 0.5, 'DENSE 1', fontsize=10, fontweight='bold')
    
    for i in range(5):
        circle = plt.Circle((1.5, layer_y - i*0.25), 0.18, color='#4ECDC4', ec='black', linewidth=1.5)
        ax.add_patch(circle)
    
    # Connections
    for i in range(5):
        for j in range(5):
            ax.plot([1.68, 3.32], [11.0 - i*0.25, layer_y - j*0.25], 
                   'k-', alpha=0.1, linewidth=0.5)
    
    ax.text(1.5, layer_y - 1.3, '128 neurons\nReLU + BatchNorm', 
           ha='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='#E0F7F6', alpha=0.7))
    
    # ===== HIDDEN LAYER 2 =====
    layer_y -= layer_spacing
    ax.text(0.3, layer_y + 0.5, 'DENSE 2', fontsize=10, fontweight='bold')
    
    for i in range(4):
        circle = plt.Circle((3.5, layer_y - i*0.25), 0.18, color='#95E1D3', ec='black', linewidth=1.5)
        ax.add_patch(circle)
    
    # Connections
    for i in range(5):
        for j in range(4):
            ax.plot([1.68, 3.32], [layer_y + 2.0 - i*0.25, layer_y - j*0.25], 
                   'k-', alpha=0.1, linewidth=0.5)
    
    ax.text(3.5, layer_y - 1.3, '64 neurons\nReLU + BatchNorm + Dropout(0.3)', 
           ha='center', fontsize=8, bbox=dict(boxstyle='round', facecolor='#F0FFF0', alpha=0.7))
    
    # ===== HIDDEN LAYER 3 =====
    layer_y -= layer_spacing
    ax.text(0.3, layer_y + 0.5, 'DENSE 3', fontsize=10, fontweight='bold')
    
    for i in range(3):
        circle = plt.Circle((5.5, layer_y - i*0.25), 0.18, color='#FFB703', ec='black', linewidth=1.5)
        ax.add_patch(circle)
    
    # Connections
    for i in range(4):
        for j in range(3):
            ax.plot([3.68, 5.32], [layer_y + 2.0 - i*0.25, layer_y - j*0.25], 
                   'k-', alpha=0.1, linewidth=0.5)
    
    ax.text(5.5, layer_y - 1.3, '32 neurons\nReLU + BatchNorm + Dropout(0.3)', 
           ha='center', fontsize=8, bbox=dict(boxstyle='round', facecolor='#FFFAE0', alpha=0.7))
    
    # ===== HIDDEN LAYER 4 =====
    layer_y -= layer_spacing
    ax.text(0.3, layer_y + 0.5, 'DENSE 4', fontsize=10, fontweight='bold')
    
    for i in range(2):
        circle = plt.Circle((7.5, layer_y - i*0.25), 0.18, color='#D62828', ec='black', linewidth=1.5)
        ax.add_patch(circle)
    
    # Connections
    for i in range(3):
        for j in range(2):
            ax.plot([5.68, 7.32], [layer_y + 2.0 - i*0.25, layer_y - j*0.25], 
                   'k-', alpha=0.1, linewidth=0.5)
    
    ax.text(7.5, layer_y - 1.1, '16 neurons\nReLU', 
           ha='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='#FFE0E0', alpha=0.7))
    
    # ===== OUTPUT LAYER =====
    layer_y -= layer_spacing + 0.3
    ax.text(0.3, layer_y + 0.5, 'OUTPUT', fontsize=10, fontweight='bold')
    
    n_output_classes = len(np.unique(y_train))
    for i in range(n_output_classes):
        circle = plt.Circle((8.5, layer_y - i*0.35), 0.2, color='#2A9D8F', ec='black', linewidth=2)
        ax.add_patch(circle)
        ax.text(8.5, layer_y - i*0.35, f'{i}', ha='center', va='center', 
               color='white', fontsize=9, fontweight='bold')
    
    # Connections
    for i in range(2):
        for j in range(n_output_classes):
            ax.plot([7.68, 8.30], [layer_y + 1.7 - i*0.25, layer_y - j*0.35], 
                   'k-', alpha=0.15, linewidth=0.8)
    
    ax.text(8.5, layer_y - 1.2, f'{n_output_classes} classes\nSoftmax', 
           ha='center', fontsize=9, bbox=dict(boxstyle='round', facecolor='#D4F1D4', alpha=0.7))
    
    # Model statistics
    stats_y = layer_y - 1.8
    n_params = dl_model.count_params()
    
    stats_text = f"""
    Model Summary:
    • Total Parameters: {n_params:,}
    • Input Features: {n_features}
    • Output Classes: {n_output_classes}
    • Trainable Layers: 5 Dense + 3 BatchNorm + 2 Dropout
    • Activation: ReLU (hidden) → Softmax (output)
    • Regularization: Batch Normalization + Dropout
    """
    
    ax.text(5, stats_y, stats_text, fontsize=9, 
           bbox=dict(boxstyle='round', facecolor='#E8E8E8', alpha=0.9),
           family='monospace', ha='center', va='top')
    
    plt.tight_layout()
    plt.savefig('reports/figures/03_neural_network_architecture.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    
    print("✓ Neural network architecture visualization saved!")
else:
    print("⚠ Deep learning model not available - skipping architecture diagram")

In [ ]:
# ============================================================================
# Visualization 4: Training History (Deep Learning Only)
# ============================================================================

if history is not None:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle('Deep Learning Model - Training History', fontsize=14, fontweight='bold')
    
    # Loss
    ax = axes[0]
    ax.plot(history.history['loss'], label='Training Loss', linewidth=2, color='#FF6B6B')
    ax.plot(history.history['val_loss'], label='Validation Loss', linewidth=2, color='#4ECDC4')
    ax.set_xlabel('Epoch', fontsize=11, fontweight='bold')
    ax.set_ylabel('Loss', fontsize=11, fontweight='bold')
    ax.set_title('Loss over Time', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Accuracy
    ax = axes[1]
    ax.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2, color='#95E1D3')
    ax.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2, color='#FFB703')
    ax.set_xlabel('Epoch', fontsize=11, fontweight='bold')
    ax.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
    ax.set_title('Accuracy over Time', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('reports/figures/04_training_history.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Training history visualization saved!")

# ============================================================================
# Visualization 5: Comprehensive Model Comparison Dashboard
# ============================================================================

fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.3)

fig.suptitle('Comprehensive Model Comparison Dashboard', fontsize=16, fontweight='bold', y=0.98)

models_list = list(metrics_dict.keys())
colors = ['#FF6B6B', '#4ECDC4', '#95E1D3'][:len(models_list)]

# Row 1: Individual metric bars
ax = fig.add_subplot(gs[0, 0])
accuracies = [metrics_dict[m]['accuracy'] for m in models_list]
ax.bar(models_list, accuracies, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
ax.set_ylabel('Accuracy', fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)
for i, v in enumerate(accuracies):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

ax = fig.add_subplot(gs[0, 1])
precisions = [metrics_dict[m]['precision'] for m in models_list]
ax.bar(models_list, precisions, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
ax.set_ylabel('Precision', fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)
for i, v in enumerate(precisions):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

ax = fig.add_subplot(gs[0, 2])
f1_scores = [metrics_dict[m]['f1'] for m in models_list]
ax.bar(models_list, f1_scores, color=colors, alpha=0.8, edgecolor='black', linewidth=2)
ax.set_ylabel('F1 Score', fontweight='bold')
ax.set_ylim([0, 1])
ax.grid(axis='y', alpha=0.3)
for i, v in enumerate(f1_scores):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Row 2: Radar / Spider chart style comparison
ax = fig.add_subplot(gs[1, :])

metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
angles = np.linspace(0, 2 * np.pi, len(metrics_names), endpoint=False).tolist()
angles += angles[:1]  # Complete the circle

ax = plt.subplot(gs[1, :], projection='polar')
for idx, model_name in enumerate(models_list):
    values = [
        metrics_dict[model_name]['accuracy'],
        metrics_dict[model_name]['precision'],
        metrics_dict[model_name]['recall'],
        metrics_dict[model_name]['f1']
    ]
    values += values[:1]  # Complete the circle
    ax.plot(angles, values, 'o-', linewidth=2, label=model_name, color=colors[idx])
    ax.fill(angles, values, alpha=0.15, color=colors[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics_names, fontsize=10, fontweight='bold')
ax.set_ylim(0, 1)
ax.set_title('Performance Metrics Comparison (Radar)', fontsize=12, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
ax.grid(True)

# Row 3: Summary statistics table
ax = fig.add_subplot(gs[2, :])
ax.axis('off')

# Create table data
table_data = []
table_data.append(['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score'])

for model_name in models_list:
    row = [
        model_name,
        f"{metrics_dict[model_name]['accuracy']:.4f}",
        f"{metrics_dict[model_name]['precision']:.4f}",
        f"{metrics_dict[model_name]['recall']:.4f}",
        f"{metrics_dict[model_name]['f1']:.4f}"
    ]
    table_data.append(row)

table = ax.table(cellText=table_data, cellLoc='center', loc='center',
                colWidths=[0.2, 0.2, 0.2, 0.2, 0.2])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2.5)

# Style header row
for i in range(5):
    table[(0, i)].set_facecolor('#4ECDC4')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Alternate row colors
for i in range(1, len(table_data)):
    for j in range(5):
        if i % 2 == 0:
            table[(i, j)].set_facecolor('#F0F0F0')
        else:
            table[(i, j)].set_facecolor('white')

ax.text(0.5, -0.15, 'Model Performance Summary', transform=ax.transAxes,
       fontsize=11, fontweight='bold', ha='center')

plt.savefig('reports/figures/05_comprehensive_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Comprehensive comparison dashboard saved!")

# ============================================================================
# Print Summary Report
# ============================================================================
print("\n" + "=" * 80)
print("MODEL COMPARISON SUMMARY REPORT")
print("=" * 80)

for model_name in models_list:
    metrics = metrics_dict[model_name]
    print(f"\n📊 {model_name.upper()}")
    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1-Score:  {metrics['f1']:.4f}")

# Find best model
best_model = max(models_list, key=lambda m: metrics_dict[m]['f1'])
print(f"\n🏆 BEST MODEL: {best_model} (F1-Score: {metrics_dict[best_model]['f1']:.4f})")

print("\n" + "=" * 80)
print("✓ All visualizations saved to: reports/figures/")
print("=" * 80)

### Visualization Files Generated

The following matplotlib visualizations were saved to `reports/figures/`:

1. **01_metrics_comparison.png** - Bar charts comparing Accuracy, Precision, Recall, and F1-Score across all three models
2. **02_confusion_matrices.png** - Confusion matrices for each model showing prediction correctness
3. **03_neural_network_architecture.png** - ⭐ **Neural network diagram** showing:
   - Input layer (14 features)
   - 4 hidden dense layers with neurons and activations
   - Batch normalization and dropout layers
   - Output layer (3 classes with softmax)
   - Total parameters and layer summary

4. **04_training_history.png** - Deep learning model training progress (loss and accuracy over epochs)
5. **05_comprehensive_comparison.png** - Dashboard with:
   - Individual metric comparisons
   - Radar chart for all metrics
   - Summary statistics table

### Model Comparison Summary

- **Random Forest (Traditional ML)**: Ensemble of decision trees, no hyperparameter tuning
- **XGBoost (Advanced Gradient Boosting)**: Boosted ensemble with optimized hyperparameters  
- **Deep Learning (Keras)**: 4-layer neural network with batch normalization and dropout regularization

### Neural Network Architecture Highlights

The deep learning model features:
- **Sequential Architecture**: 128 → 64 → 32 → 16 → output layers
- **Batch Normalization**: Applied after each hidden layer for training stability
- **Dropout Regularization**: 30% dropout in early layers to prevent overfitting
- **ReLU Activation**: Fast, efficient activation function for hidden layers
- **Softmax Output**: Multi-class probability distribution

### Performance Interpretation

- **Accuracy**: Overall prediction correctness across all classes
- **Precision**: When the model predicts positive, how often is it correct
- **Recall**: Of actual positives, how many did the model correctly identify
- **F1-Score**: Harmonic mean of precision and recall (best single metric)

### Recommendations

1. **For Production**: Choose model with best F1-score to balance precision/recall
2. **For Real-time**: Random Forest offers fastest inference (no batch processing needed)
3. **For Explainability**: Random Forest and shallow XGBoost are more interpretable than DL
4. **For Accuracy**: Deep learning often achieves best performance but needs more data
5. **For Tuning**: Use validation metrics to adjust hyperparameters before final testing

## 7. Key Insights & Next Steps

## 6. Compare Model Results with Subplots (Advanced Comparison)

## 5. Visualize Neural Network Architecture (Deep Learning Model)

## 4. Visualize Model Performance Metrics

## 3. Train Three Models (Traditional ML, Advanced ML, Deep Learning)

In [ ]:
# Load metadata to understand data structure
metadata_path = PROJECT_ROOT / "data" / "processed" / "v1" / "incident_remediation_metadata.json"

try:
    with open(metadata_path, 'r') as f:
        metadata = json.load(f)
    
    n_features = metadata['train_shape'][1]
    n_samples_train = metadata['train_shape'][0]
    n_samples_val = metadata['val_shape'][0]
    n_samples_test = metadata['test_shape'][0]
    
    print(f"✓ Dataset Info from Metadata:")
    print(f"  Training samples: {n_samples_train}")
    print(f"  Validation samples: {n_samples_val}")
    print(f"  Test samples: {n_samples_test}")
    print(f"  Features: {n_features}")
    print(f"  Features: {metadata['feature_columns']}")
    
except FileNotFoundError:
    print("⚠ Metadata file not found. Using default configuration.")
    n_features = 14
    n_samples_train = 34988
    n_samples_val = 7497
    n_samples_test = 7498

# Load actual data
data_dir = PROJECT_ROOT / "data" / "processed" / "v1"
print(f"\nLoading data from: {data_dir}")

try:
    X_train = pd.read_csv(data_dir / "X_train.csv").values
    X_val = pd.read_csv(data_dir / "X_val.csv").values
    X_test = pd.read_csv(data_dir / "X_test.csv").values
    
    y_train = pd.read_csv(data_dir / "y_train.csv").values.ravel()
    y_val = pd.read_csv(data_dir / "y_val.csv").values.ravel()
    y_test = pd.read_csv(data_dir / "y_test.csv").values.ravel()
    
    print(f"✓ Data loaded successfully!")
    print(f"  X_train shape: {X_train.shape}")
    print(f"  X_val shape: {X_val.shape}")
    print(f"  X_test shape: {X_test.shape}")
    
except FileNotFoundError as e:
    print(f"⚠ Data files not found: {e}")
    print("Creating synthetic data for demonstration...")
    
    n_train, n_val, n_test = 2000, 500, 500
    n_features = 14
    n_classes = 3
    
    X_train = np.random.randn(n_train, n_features)
    X_val = np.random.randn(n_val, n_features)
    X_test = np.random.randn(n_test, n_features)
    
    y_train = np.random.randint(0, n_classes, n_train)
    y_val = np.random.randint(0, n_classes, n_val)
    y_test = np.random.randint(0, n_classes, n_test)
    
    print(f"✓ Synthetic data created")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print(f"\n✓ Data preprocessing complete!")
print(f"  Number of classes: {len(np.unique(y_test))}")

## 2. Load and Prepare Data

In [ ]:
import sys
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from typing import Dict, Any, Tuple

# ML & Data Processing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
import xgboost as xgb

# Deep Learning
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    from tensorflow.keras.models import Model
    print(f"✓ TensorFlow {tf.__version__} loaded")
except ImportError:
    print("⚠ TensorFlow not available. DL model will be skipped.")

# Add project to path
PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42) if 'tf' in dir() else None

print("✓ All libraries imported successfully!")

## 1. Import Required Libraries

# Model Comparison Visualizations with Matplotlib

Comprehensive guide for training three models and visualizing their performance:
- **Traditional ML**: Logistic Regression or Random Forest
- **Advanced ML**: XGBoost (Gradient Boosting)
- **Deep Learning**: Neural Network (Keras/TensorFlow)

Plus: **Neural Network Architecture Diagram** specifically for the DL model